# Sub-Category: PGA-UNet vs U-Net 2D

> **Lưu ý về thuật ngữ:** nhóm **TOP-DICE** / **BOTTOM-DICE** được trích xuất **hậu nghiệm** dựa trên chính kết quả dự đoán (Dice) của U-Net trên toàn bộ tập test — **không phải** gán nhãn độ khó/dễ của ảnh từ trước. Đây là ảnh mà U-Net (baseline) tự dự đoán tốt nhất / kém nhất, sau đó cùng bộ ảnh đó được đưa cho PGA-UNet dự đoán lại để so sánh công bằng trên cùng baseline-derived subset.

| Section | Nội dung |
|---|---|
| **0+1** | U-Net chạy **toàn bộ** → sort Dice → **easy_stems → TOP-DICE (top-50)** + **hard_stems → BOTTOM-DICE (bot-50)** — dùng chung cho cả 2 model; **metrics N=50**, **visualize top-10** |
| **3** | PGA-UNet: metrics N=50 stems (số poly ≥ 50), visualize polygon từ **10 stems đầu** |
| **Agg** | Bảng tổng hợp 2 model × TOP-DICE / BOTTOM-DICE (N=50) + bar chart |

Visualization 4 cột: **Ảnh + Prompt BBox** | **GT** | **Dự đoán** *(Dice/IoU/CBL)* | **Overlap** *(Pre/Rec/HD95)*

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 1 — SETUP (clone, dataset, checkpoints, utilities)
# ══════════════════════════════════════════════════════
%cd /content
import os, sys, cv2, json as _json
import numpy as np, torch
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle as Rect
!pip install -q gdown tqdm opencv-python matplotlib scipy
import gdown

REPO='https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git'
for d,b in [('unet-repo',None),('pga-repo','TN_B_ON')]:
    if not os.path.exists(f'/content/{d}'):
        br=f'-b {b} ' if b else ''
        os.system(f'git clone -q {br}{REPO} /content/{d}')
    print(f'  ✅ {d}')

if not os.path.exists('/content/dataset_BTXRD'):
    gdown.download('https://drive.google.com/uc?id=1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW',
                   '/content/dataset_BTXRD.zip', quiet=False)
    os.system('unzip -q /content/dataset_BTXRD.zip -d /content/')
for d in ['unet-repo','pga-repo']:
    if not os.path.exists(f'/content/{d}/dataset_BTXRD'):
        os.system(f'cp -r /content/dataset_BTXRD /content/{d}/dataset_BTXRD')
print('  ✅ Dataset ready')

os.makedirs('/content/checkpoints',exist_ok=True)
for cid,fn in [('1hKyyw8WvN6WRyzjDE50upWI0NIDJbSMu','unet_best.pth'),
               ('1sSoQ8SObteWETuKSsGGhBASPtXQz9JbY', 'pga_unet_expB_best.pth')]:
    p=f'/content/checkpoints/{fn}'
    if not os.path.exists(p): gdown.download(f'https://drive.google.com/uc?id={cid}',p,quiet=False)
    print(f'  ✅ {fn}  {os.path.getsize(p)//1024}KB')

DEVICE,IMG_SIZE,ZOOM_R=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),512,0.30
IMG_DIR ='/content/unet-repo/dataset_BTXRD/test/images'
MASK_DIR='/content/unet-repo/dataset_BTXRD/test/masks'
JSON_DIR='/content/pga-repo/dataset_BTXRD/test/annotations'
KEYS=['dice','iou','precision','recall','hd95','cbl']

def calc_metrics(prob,gt,eps=1e-6):
    pm=(prob>0.5).astype(np.float32); gm=(gt>0.5).astype(np.float32)
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    p,g=pm.astype(bool),gm.astype(bool); hd95=float(IMG_SIZE)
    if p.any() and g.any():
        pe=p^binary_erosion(p); ge=g^binary_erosion(g)
        d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
        if len(d1) and len(d2): hd95=float(max(np.percentile(d1,95),np.percentile(d2,95)))
    if gm.sum()==0 or pm.sum()==0: cbl=0.
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)),iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=(float(tp/(tp+fp)) if (tp+fp)>0 else 0.),recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95,cbl=cbl,mask=pm)

def plateau_heatmap(bbox,S=512):
    x1=max(0,int(bbox[0])-5);y1=max(0,int(bbox[1])-5)
    x2=min(S,int(bbox[2])+5);y2=min(S,int(bbox[3])+5)
    hm=np.zeros((S,S),dtype=np.float32)
    if x2>x1 and y2>y1: hm[y1:y2,x1:x2]=1.0; hm=cv2.GaussianBlur(hm,(31,31),0)
    return hm

def print_metrics(label,mlist):
    m={k:np.mean([r[k] for r in mlist]) for k in KEYS}
    print(f"  {label:<20} Dice={m['dice']:.4f} IoU={m['iou']:.4f} Pre={m['precision']:.4f}"
          f" Rec={m['recall']:.4f} HD95={m['hd95']:.2f} CBL={m['cbl']:.4f}")
    return m

def visualize(records, title, n=15, show_bbox=False):
    recs=records[:n]; nr=len(recs)
    fig,axes=plt.subplots(nr,4,figsize=(20,4.5*nr),squeeze=False)
    fig.suptitle(title,fontsize=13,fontweight='bold',y=1.005)
    for j,t in enumerate(['Ảnh gốc','Ground Truth',
                           'Dự đoán\n(Dice / IoU / CBL)',
                           'Overlap\n(Pre / Rec / HD95)']):
        axes[0,j].set_title(t,fontsize=9,fontweight='bold')
    for row,rec in enumerate(recs):
        gray=np.clip(rec['img_np']*255,0,255).astype(np.uint8)
        rgb=cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB)
        gt=rec['gt']; pm=rec['m']['mask']
        def ov(mask,clr,a=0.5):
            o=rgb.copy().astype(np.float32); o[mask>0.5]=o[mask>0.5]*(1-a)+np.array(clr)*a
            return np.clip(o,0,255).astype(np.uint8)
        diff=rgb.copy(); pb,gb=pm>0.5,gt>0.5
        diff[gb&~pb]=[30,180,30]; diff[pb&~gb]=[220,60,60]; diff[pb&gb]=[220,200,0]
        ax=axes[row,0]; ax.imshow(gray,cmap='gray')
        if show_bbox:
            for bx in rec.get('bboxes',[]):
                ax.add_patch(Rect((bx[0],bx[1]),bx[2]-bx[0],bx[3]-bx[1],lw=1.5,edgecolor='lime',facecolor='none'))
        ax.set_ylabel(rec['stem'],fontsize=7,rotation=0,labelpad=75,va='center'); ax.axis('off')
        axes[row,1].imshow(ov(gt,[30,120,255])); axes[row,1].axis('off')
        m=rec['m']
        axes[row,2].imshow(ov(pm,[220,60,60]))
        axes[row,2].set_title(f"Dice={m['dice']:.3f}  IoU={m['iou']:.3f}  CBL={m['cbl']:.3f}",
                              fontsize=8,color='darkred',pad=2); axes[row,2].axis('off')
        axes[row,3].imshow(diff)
        axes[row,3].set_title(f"Pre={m['precision']:.3f}  Rec={m['recall']:.3f}  HD95={m['hd95']:.1f}px",
                              fontsize=8,color='saddlebrown',pad=2); axes[row,3].axis('off')
    plt.tight_layout()
    fname=title.split('|')[0].strip()[:6].lower()+'_'+title.split('|')[-1].strip()[:4].lower()+'.png'
    plt.savefig(fname,dpi=100,bbox_inches='tight'); plt.show(); print(f'  ✅ {fname}')

# Load dataset & image stems
sys.path.insert(0,'/content/unet-repo')
from dataset import BTXRD_Dataset
test_ds=BTXRD_Dataset(image_dir=IMG_DIR,mask_dir=MASK_DIR,img_size=IMG_SIZE,is_train=False)
# UNet dataset dùng os.listdir() không sort → phải sort lại để khớp với img_stems
test_ds.images=sorted(test_ds.images)

img_stems=[os.path.splitext(f)[0] for f in test_ds.images]  # lấy stem từ đúng thứ tự dataset

all_raw=[]
for idx,(img_t,mask_t) in enumerate(DataLoader(test_ds,batch_size=1,shuffle=False)):
    all_raw.append({'idx':idx,
                    'img_np':img_t[0,0].numpy().copy(),   # [0,1]
                    'gt':mask_t[0,0].numpy().copy(),
                    'img_t':img_t.clone(),
                    'stem':img_stems[idx]})
SEC={}
print(f'\n✅ Setup xong | {len(all_raw)} samples | device={DEVICE}')
print(f'   Ví dụ stems: {img_stems[:3]} ...')


In [ ]:
# ── Redefine visualize() — TP=xanh lá / FP=đỏ / FN=xanh dương + legend ──────
from matplotlib.patches import Patch, Rectangle as Rect
from IPython.display import display as _ipy_display

def visualize(records, title, n=15, show_bbox=False):
    recs = records[:n]; nr = len(recs)
    fig, axes = plt.subplots(nr, 4, figsize=(20, 4.5 * nr), squeeze=False)
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.005)
    for j, t in enumerate(['Ảnh gốc', 'Ground Truth',
                            'Dự đoán\n(Dice / IoU / CBL)',
                            'TP/FP/FN\n(Pre / Rec / HD95)']):
        axes[0, j].set_title(t, fontsize=9, fontweight='bold')
    for row, rec in enumerate(recs):
        gray = np.clip(rec['img_np'] * 255, 0, 255).astype(np.uint8)
        rgb  = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
        gt   = rec['gt']; pm = rec['m']['mask']

        def ov(mask, clr, a=0.5):
            o = rgb.copy().astype(np.float32)
            o[mask > 0.5] = o[mask > 0.5] * (1 - a) + np.array(clr) * a
            return np.clip(o, 0, 255).astype(np.uint8)

        # TP=xanh lá, FP=đỏ, FN=xanh dương (cùng chuẩn PGA)
        pb, gb = pm > 0.5, gt > 0.5
        bg_f = rgb.astype(np.float32) / 255. * 0.25
        inter = bg_f.copy()
        inter[..., 1] = np.clip(inter[..., 1] + (pb & gb).astype(float)  * 0.9, 0, 1)
        inter[..., 0] = np.clip(inter[..., 0] + (pb & ~gb).astype(float) * 1.0, 0, 1)
        inter[..., 2] = np.clip(inter[..., 2] + (~pb & gb).astype(float) * 1.0, 0, 1)
        inter_u8 = (inter * 255).astype(np.uint8)

        ax = axes[row, 0]; ax.imshow(gray, cmap='gray')
        if show_bbox:
            for bx in rec.get('bboxes', []):
                ax.add_patch(Rect((bx[0], bx[1]), bx[2] - bx[0], bx[3] - bx[1],
                                  lw=1.5, edgecolor='lime', facecolor='none'))
        ax.set_ylabel(rec['stem'], fontsize=7, rotation=0, labelpad=75, va='center')
        ax.axis('off')
        axes[row, 1].imshow(ov(gt, [30, 120, 255])); axes[row, 1].axis('off')
        m = rec['m']
        axes[row, 2].imshow(ov(pm, [220, 60, 60]))
        axes[row, 2].set_title(f"Dice={m['dice']:.3f}  IoU={m['iou']:.3f}  CBL={m['cbl']:.3f}",
                               fontsize=8, color='darkred', pad=2); axes[row, 2].axis('off')
        axes[row, 3].imshow(inter_u8)
        axes[row, 3].set_title(f"Pre={m['precision']:.3f}  Rec={m['recall']:.3f}  HD95={m['hd95']:.1f}px",
                               fontsize=8, color='saddlebrown', pad=2); axes[row, 3].axis('off')
    fig.legend(
        handles=[Patch(facecolor='green', label='TP'),
                 Patch(facecolor='red',   label='FP'),
                 Patch(facecolor='blue',  label='FN')],
        loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.004)
    )
    plt.tight_layout()
    fname = title.split('|')[0].strip()[:6].lower() + '_' + title.split('|')[-1].strip()[:4].lower() + '.png'
    plt.savefig(fname, dpi=100, bbox_inches='tight')
    _ipy_display(fig); plt.close(fig)
    print(f'  ✅ {fname}')

print('✅ visualize() redefined — TP=green / FP=red / FN=blue + legend')

In [ ]:
# ══════════════════════════════════════════════════════
# SECTION 0+1 — U-Net 2D
# Chạy toàn bộ → sort Dice → TOP-DICE / BOTTOM-DICE stems
# (phân nhóm HẬU NGHIỆM theo Dice của chính U-Net, KHÔNG phải độ khó/dễ ảnh gán trước)
# Metrics: N_METRIC=50 | Visualize: N_SHOW=10
# ══════════════════════════════════════════════════════
N_METRIC=50   # số ảnh dùng để tính độ đo
N_SHOW  =10   # số ảnh hiển thị

%cd /content/unet-repo
for _k in list(sys.modules.keys()):
    if 'models' in _k: del sys.modules[_k]
if '/content/unet-repo' not in sys.path:
    sys.path.insert(0, '/content/unet-repo')
else:
    sys.path.remove('/content/unet-repo'); sys.path.insert(0, '/content/unet-repo')

from models.networks.unet_2D import unet_2D
unet=unet_2D(in_channels=1,n_classes=1).to(DEVICE)
unet.load_state_dict(torch.load('/content/checkpoints/unet_best.pth',map_location=DEVICE,weights_only=True))
unet.eval(); print('✅ U-Net loaded')

all_unet_res=[]
with torch.no_grad():
    for s in tqdm(all_raw,'UNet-All'):
        prob=torch.sigmoid(unet(s['img_t'].to(DEVICE)))[0,0].cpu().numpy()
        all_unet_res.append(calc_metrics(prob,s['gt']))
del unet

sorted_all=sorted(range(len(all_raw)),key=lambda i:all_unet_res[i]['dice'],reverse=True)
easy_idx=sorted_all[:N_METRIC]
hard_idx=sorted_all[-N_METRIC:][::-1]

easy_stems=[all_raw[i]['stem'] for i in easy_idx]   # TOP-DICE:    U-Net dự đoán tốt nhất
hard_stems =[all_raw[i]['stem'] for i in hard_idx]  # BOTTOM-DICE: U-Net dự đoán kém nhất

print(f'\n  TOP-DICE top-{N_METRIC} (U-Net dự đoán tốt nhất)    Dice: {all_unet_res[easy_idx[0]]["dice"]:.4f} → {all_unet_res[easy_idx[-1]]["dice"]:.4f}')
print(f'  BOTTOM-DICE bot-{N_METRIC} (U-Net dự đoán kém nhất)  Dice: {all_unet_res[hard_idx[0]]["dice"]:.4f} → {all_unet_res[hard_idx[-1]]["dice"]:.4f}')
print(f'\neasy_stems  (TOP-DICE)    = {easy_stems}')
print(f'hard_stems  (BOTTOM-DICE) = {hard_stems}')

bar='='*70
print(f'\n{bar}\n  SECTION 1 — U-Net  (metrics N={N_METRIC}, show N={N_SHOW})\n{bar}')
SEC['unet']={
    'easy': print_metrics(f'TOP-DICE (N={N_METRIC})', [all_unet_res[i] for i in easy_idx]),
    'hard': print_metrics(f'BOTTOM-DICE (N={N_METRIC})', [all_unet_res[i] for i in hard_idx])
}

stem2idx={all_raw[i]['stem']:i for i in range(len(all_raw))}
recs_e=[dict(**all_raw[stem2idx[st]], m=all_unet_res[stem2idx[st]]) for st in easy_stems]
recs_h=[dict(**all_raw[stem2idx[st]], m=all_unet_res[stem2idx[st]]) for st in hard_stems]
visualize(recs_e, f'U-Net | TOP-DICE — top-{N_SHOW} / {N_METRIC} ảnh Dice cao nhất (theo U-Net)', n=N_SHOW)
visualize(recs_h, f'U-Net | BOTTOM-DICE — bottom-{N_SHOW} / {N_METRIC} ảnh Dice thấp nhất (theo U-Net)', n=N_SHOW)

In [ ]:
# ══════════════════════════════════════════════════════
# SECTION 3 — PGA-UNet  (IMAGE-LEVEL merging)
# Mỗi ảnh: chạy tất cả polygon → max-merge prob + GT union → metric ảnh
# ══════════════════════════════════════════════════════

_EASY = []   # paste easy_stems từ output cell-2 khi chạy ĐỘC LẬP
_HARD  = []
_N_METRIC, _N_SHOW = 50, 10
if 'easy_stems' not in dir() or not easy_stems: easy_stems, hard_stems = _EASY, _HARD
if 'N_METRIC'   not in dir(): N_METRIC = _N_METRIC
if 'N_SHOW'     not in dir(): N_SHOW   = _N_SHOW
assert easy_stems and hard_stems, "❌ Paste easy_stems/hard_stems từ cell-2 vào _EASY/_HARD!"

%cd /content/pga-repo
for _k in list(sys.modules.keys()):
    if 'models' in _k: del sys.modules[_k]
if '/content/pga-repo' not in sys.path:
    sys.path.insert(0, '/content/pga-repo')
else:
    sys.path.remove('/content/pga-repo'); sys.path.insert(0, '/content/pga-repo')

from models.networks.prompt_unet_2D import PGA_UNet
import importlib.util, torch
_spec=importlib.util.spec_from_file_location('pga_ds_mod','/content/pga-repo/dataset.py')
_mod=importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_mod)
PGA_Dataset=_mod.BTXRD_Dataset

pga=PGA_UNet(in_channels=1,n_classes=1,use_encoder_prompt=True).to(DEVICE)
pga.load_state_dict(torch.load('/content/checkpoints/pga_unet_expB_best.pth',
                                map_location=DEVICE,weights_only=True))
pga.eval(); print('✅ PGA-UNet loaded')

pga_ds=PGA_Dataset(image_dir=IMG_DIR, json_dir=JSON_DIR,
                   img_size=IMG_SIZE, is_train=False, prompt_mode='zoom_out')

stem_to_pga={}
for i,(img_name,_) in enumerate(pga_ds.all_samples):
    stem_to_pga.setdefault(os.path.splitext(img_name)[0],[]).append(i)


def calc_metrics_img(prob, gt, eps=1e-6, IMG_S=512):
    """Image-level: NO LCC (nhiều polygon). GT union + max-merge prob."""
    pm = (prob > 0.5).astype(np.float32)
    gm = (gt   > 0.5).astype(np.float32)
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    p, g = pm.astype(bool), gm.astype(bool); hd95 = float(IMG_S)
    if p.any() and g.any():
        from scipy.ndimage import binary_erosion, distance_transform_edt
        pe=p^binary_erosion(p); ge=g^binary_erosion(g)
        d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
        if len(d1) and len(d2): hd95=float(max(np.percentile(d1,95),np.percentile(d2,95)))
    if gm.sum()==0 or pm.sum()==0: cbl=0.
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=(float(tp/(tp+fp)) if (tp+fp)>0 else 0.),   recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl, mask=pm)


def infer_pga_image(stem):
    """Chạy PGA trên tất cả polygon của stem, trả về img_np/gt_union/prob_max/prompts."""
    idxs = stem_to_pga.get(stem, [])
    if not idxs: return None
    gt_union = None; prob_max = None; prompts = []; img_np = None
    with torch.no_grad():
        for idx in idxs:
            img_t, gt_t, hm_t = pga_ds[idx]
            if img_np is None: img_np = img_t[0].numpy()
            gt_np = gt_t[0].numpy(); hm_np = hm_t[0].numpy()
            prob  = torch.sigmoid(pga(img_t.unsqueeze(0).to(DEVICE),
                                      hm_t.unsqueeze(0).to(DEVICE)))[0,0].cpu().numpy()
            gt_union = gt_np if gt_union is None else np.maximum(gt_union, gt_np)
            prob_max = prob  if prob_max is None else np.maximum(prob_max, prob)
            prompts.append(hm_np)
    return dict(stem=stem, img_np=img_np, gt=gt_union, prob=prob_max,
                prompts=prompts, n_samples=len(idxs))

print('  Running PGA inference per-image ...')
easy_img = [r for r in (infer_pga_image(s) for s in easy_stems) if r]
hard_img  = [r for r in (infer_pga_image(s) for s in hard_stems)  if r]
del pga

for r in easy_img + hard_img:
    r['m'] = calc_metrics_img(r['prob'], r['gt'])

bar='='*70
print(f'\n{bar}\n  SECTION 3 — PGA-UNet  (image-level, N={N_METRIC} stems, cùng nhóm TOP-DICE/BOTTOM-DICE lấy từ U-Net)\n{bar}')
SEC['pga'] = {
    'easy': print_metrics(f"TOP-DICE ({len(easy_img)} ảnh)", [r['m'] for r in easy_img]),
    'hard': print_metrics(f"BOTTOM-DICE ({len(hard_img)} ảnh)",  [r['m'] for r in hard_img]),
    'n_easy': len(easy_img), 'n_hard': len(hard_img),
}

def visualize_img(records, title, n=10):
    """Image-level vis: merged pred + GT union."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    from IPython.display import display as _disp
    recs = records[:n]; nr = len(recs)
    if nr == 0: return
    fig, axes = plt.subplots(nr, 5, figsize=(20, 4*nr), squeeze=False)
    fig.suptitle(title, fontsize=12, fontweight='bold', y=1.002)
    for j,t in enumerate(['Ảnh gốc','Prompts (merged)','Dự đoán (merged)','GT (union)','TP/FP/FN']):
        axes[0,j].set_title(t, fontsize=9, fontweight='bold')
    for row, rec in enumerate(recs):
        img_np = np.clip((rec['img_np']*0.5+0.5), 0, 1)
        gt     = (rec['gt']>0.5).astype(float)
        pred   = (rec['prob']>0.5).astype(float)
        pm_merged = np.max(np.stack(rec['prompts'],0),0) if rec.get('prompts') else np.zeros_like(img_np)
        tp=(pred*gt).sum(); fp=(pred*(1-gt)).sum(); fn=((1-pred)*gt).sum(); e=1e-6
        dice=float((2*tp+e)/(2*tp+fp+fn+e)); iou=float((tp+e)/(tp+fp+fn+e))
        pre=float((tp+e)/(tp+fp+e));         rec_=float((tp+e)/(tp+fn+e))
        n_poly = rec.get('n_samples', '?')
        bg = np.stack([img_np]*3,-1)
        axes[row,0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
        axes[row,0].set_ylabel(f"{rec['stem']} [{n_poly}p]\nDice={dice:.3f}", fontsize=7)
        axes[row,1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
        axes[row,1].imshow(pm_merged, cmap='hot', alpha=0.4, vmin=0, vmax=1)
        pr_ov=bg.copy(); pr_ov[...,0]=np.clip(pr_ov[...,0]+pred*0.55,0,1); pr_ov[...,1]=np.clip(pr_ov[...,1]-pred*0.2,0,1)
        axes[row,2].imshow(pr_ov); axes[row,2].set_title(f'Dice={dice:.3f} IoU={iou:.3f}',fontsize=7,color='darkred',pad=2)
        gt_ov=bg.copy(); gt_ov[...,1]=np.clip(gt_ov[...,1]+gt*0.55,0,1); gt_ov[...,0]=np.clip(gt_ov[...,0]-gt*0.2,0,1)
        axes[row,3].imshow(gt_ov)
        inter=bg.copy()*0.25
        inter[...,1]=np.clip(inter[...,1]+pred*gt*0.9,0,1)
        inter[...,0]=np.clip(inter[...,0]+pred*(1-gt)*1.0,0,1)
        inter[...,2]=np.clip(inter[...,2]+(1-pred)*gt*1.0,0,1)
        axes[row,4].imshow(inter); axes[row,4].set_title(f'Pre={pre:.3f} Rec={rec_:.3f}',fontsize=7,color='saddlebrown',pad=2)
        for ax in axes[row]: ax.axis('off')
    fig.legend(handles=[Patch(facecolor='green',label='TP'),Patch(facecolor='red',label='FP'),Patch(facecolor='blue',label='FN')],
               loc='lower center',ncol=3,fontsize=8,bbox_to_anchor=(0.5,-0.004))
    plt.tight_layout(); _disp(fig); plt.close(fig)

visualize_img(easy_img[:N_SHOW], f'PGA | TOP-DICE — {N_SHOW} ảnh đầu (image-level merged, nhóm U-Net Dice cao nhất)')
visualize_img(hard_img[:N_SHOW],  f'PGA | BOTTOM-DICE — {N_SHOW} ảnh đầu (image-level merged, nhóm U-Net Dice thấp nhất)')

In [ ]:
# ══════════════════════════════════════════════════════
# CELL TỔNG HỢP — So sánh 2 model trên cùng 50+50 ảnh
# Tất cả model đều tính IMAGE-LEVEL (UNet/PGA: UNet whole-image, PGA max-merge per polygon)
# Nhóm TOP-DICE/BOTTOM-DICE lấy HẬU NGHIỆM theo Dice của U-Net (không phải độ khó ảnh gán trước)
# ══════════════════════════════════════════════════════
import csv, os

GRP_LABEL = {'easy': 'TOP-DICE', 'hard': 'BOTTOM-DICE'}

N_EASY = len(easy_stems)   # = 50
N_HARD = len(hard_stems)   # = 50

bar = '═'*90
print(f'\n{bar}')
print(f'  BẢNG TỔNG KẾT  |  {N_EASY} {GRP_LABEL["easy"]} + {N_HARD} {GRP_LABEL["hard"]} ảnh (image-level)')
print(f'  (phân nhóm hậu nghiệm theo Dice của U-Net trên toàn tập test — không phải độ khó ảnh gán trước)')
print(f'{bar}')
print(f"  {'Model':<14} {'Nhóm':<12} {'N':>6} {'Dice':>7} {'IoU':>7} {'Pre':>7} {'Rec':>7} {'HD95':>8} {'CBL':>7}")
print(f"  {'-'*78}")

csv_rows = []
for model, key in [('U-Net', 'unet'),
 ('PGA-UNet', 'pga')]:
    for grp in ['easy', 'hard']:
        m = SEC[key][grp]
        n = N_EASY if grp == 'easy' else N_HARD
        print(f"  {model:<14} {GRP_LABEL[grp]:<12} {n:>5} img"
              f" {m['dice']:>7.4f} {m['iou']:>7.4f} {m['precision']:>7.4f}"
              f" {m['recall']:>7.4f} {m['hd95']:>8.2f} {m['cbl']:>7.4f}")
        csv_rows.append([model, GRP_LABEL[grp], n] + [f"{m[k]:.4f}" for k in KEYS])
    print(f"  {'-'*78}")

print(f"  {'Δ PGA−UNet':<14} {GRP_LABEL['easy']:<12} {'':<6}"
      f" {SEC['pga']['easy']['dice']-SEC['unet']['easy']['dice']:>+7.4f}")
print(f"  {'':14} {GRP_LABEL['hard']:<12} {'':<6}"
      f" {SEC['pga']['hard']['dice']-SEC['unet']['hard']['dice']:>+7.4f}")
print(bar)

os.makedirs('results', exist_ok=True)
with open('results/subcat_pga_vs_baseline.csv', 'w', newline='') as f:
    csv.writer(f).writerows([['model', 'group', 'N'] + KEYS] + csv_rows)
print('→ CSV: results/subcat_pga_vs_baseline.csv')

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(2); w = 0.25
for i, (label, key, clr) in enumerate([('U-Net', 'unet', '#6699cc'),
                                         ('PGA-UNet', 'pga', '#ef5350')]):
    vals = [SEC[key]['easy']['dice'], SEC[key]['hard']['dice']]
    bars = ax.bar(x + (i-1)*w, vals, w, label=label, color=clr, edgecolor='white')
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, v+0.012, f'{v:.3f}',
                ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'{GRP_LABEL["easy"]}  ({N_EASY} ảnh)', f'{GRP_LABEL["hard"]}  ({N_HARD} ảnh)'], fontsize=10)
ax.set_ylabel('Dice'); ax.set_ylim(0, 1.15)
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
ax.set_title('PGA-UNet vs Baseline — Top-Dice vs Bottom-Dice (image-level)\n'
             '(phân nhóm hậu nghiệm theo Dice của U-Net, không phải độ khó ảnh)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('results/subcat_baseline_bar.png', dpi=130, bbox_inches='tight'); plt.show()
print('→ Plot: results/subcat_baseline_bar.png')